# SAiDL Core ML + Mechanistic Interpretability Notebook

This notebook provides a minimal workflow for running the Core ML transformer training, extracting activation data, and evaluating the Sparse Autoencoder (SAE) + quantization pipeline. It is intended as a reproducible demo for the repository.

In [ ]:
# Install required packages if not already available
!pip install torch torchvision torchaudio transformers datasets wandb omegaconf hydra-core tqdm

In [ ]:
# Verify repository structure and import new MI modules
import os
from pathlib import Path
repo_root = Path('.')
print('Repo root:', repo_root.resolve())
print('Core ML train script exists:', (repo_root / 'core_ml' / 'train' / 'train.py').exists())
print('MI train utils exists:', (repo_root / 'mechanistic_interpretability' / 'train' / 'sae_trainer.py').exists())

## Recommended Local Commands

Use the repository scripts from the terminal rather than running long training sessions in the notebook to preserve resources. Example commands: 

```bash
python core_ml/train/train.py attention=vanilla_mha positional=sinusoidal model.name=baseline_transformer
python mechanistic_interpretability/pipeline/quantize.py
```

# SAiDL Summer Assignment 2026 — Core ML

**Execution order:**
1. Setup (run all cells in Section 0 once)
2. Core ML — train each experiment (Section 1)
3. Benchmark all checkpoints (Section 2)

Every cell prints a clear ✅ / ❌ status and the full traceback on failure.

## 0  One-time Setup

In [ ]:
# ── 0A  Clone / pull repo ─────────────────────────────────────────────────────
import os, sys, traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'

try:
    if not os.path.exists(REPO):
        ret = os.system(f'git clone https://github.com/VvS-2403/SAiDL-Summer-Assignment-2026.git {REPO}')
        assert ret == 0, f'git clone failed with exit code {ret}'
    else:
        ret = os.system(f'cd {REPO} && git pull')
        assert ret == 0, f'git pull failed with exit code {ret}'

    os.chdir(REPO)
    sys.path.insert(0, REPO)
    print(f'✅ Repo ready at {REPO}')
except Exception as e:
    print(f'❌ FAILED: {e}')
    traceback.print_exc()

In [ ]:
# ── 0B  Install dependencies ──────────────────────────────────────────────────
import subprocess, sys

packages = [
    'transformers', 'datasets', 'wandb', 'hydra-core',
    'omegaconf', 'einops', 'tqdm', 'scikit-learn', 'matplotlib', 'numpy'
]

try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('❌ pip install failed:')
        print(result.stderr)
    else:
        print('✅ Packages installed')
except Exception as e:
    print(f'❌ FAILED: {e}')
    import traceback; traceback.print_exc()

In [ ]:
# ── 0C  Create / verify all config files ─────────────────────────────────────
#
# BUGS FIXED vs original notebook:
#   BUG 1: relu.yaml had name: "relu"  → must be "relu_attention" (matches train.py dispatch)
#   BUG 2: sparse.yaml had name: "sparse" → must be "sparse_attention"
#   BUG 3: relative.yaml had max_len key → must be max_relative_distance (matches RelativePositionalBias)
#   BUG 4: hybrid.yaml had flat hybrid_kernel_size key → must be nested hybrid: {type, conv_kernel_size}
#          (train.py reads cfg.model.hybrid.type and cfg.model.hybrid.conv_kernel_size)

import os, textwrap, traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'
errors = []

def write_config(path, content):
    """Always write (overwrite) — ensures stale wrong configs are replaced."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(textwrap.dedent(content).lstrip())
    print(f'  wrote {os.path.relpath(path, REPO)}')

try:
    BASE = f'{REPO}/core_ml/configs'

    # ── Attention configs ──────────────────────────────────────────────────
    write_config(f'{BASE}/attention/vanilla.yaml',
        'name: "vanilla_mha"\nnum_heads: 8\ndropout: 0.1\nis_causal: true\n')

    write_config(f'{BASE}/attention/gqa.yaml',
        'name: "gqa"\nnum_heads: 8\nnum_kv_heads: 2\ndropout: 0.1\nis_causal: true\n')

    write_config(f'{BASE}/attention/sliding_window.yaml',
        'name: "sliding_window"\nnum_heads: 8\nwindow_size: 256\ndropout: 0.1\nis_causal: true\n')

    # FIX BUG 1: was name: "relu"
    write_config(f'{BASE}/attention/relu.yaml',
        'name: "relu_attention"\nnum_heads: 8\ndropout: 0.1\nis_causal: true\n')

    # FIX BUG 2: was name: "sparse"
    write_config(f'{BASE}/attention/sparse.yaml',
        'name: "sparse_attention"\nnum_heads: 8\nlocal_window: 64\nstride: 64\ndropout: 0.1\nis_causal: true\n')

    # ── Positional configs ─────────────────────────────────────────────────
    write_config(f'{BASE}/positional/sinusoidal.yaml',
        'name: "sinusoidal"\nmax_len: 1024\nbase: 10000\n')

    write_config(f'{BASE}/positional/rope.yaml',
        'name: "rope"\nmax_len: 4096\nbase: 10000.0\n')

    write_config(f'{BASE}/positional/alibi.yaml',
        'name: "alibi"\n')

    # FIX BUG 3: was max_len: 1024; RelativePositionalBias reads max_relative_distance
    write_config(f'{BASE}/positional/relative.yaml',
        'name: "relative"\nmax_relative_distance: 128\n')

    # ── Model configs ──────────────────────────────────────────────────────
    write_config(f'{BASE}/model/transformer.yaml',
        'name: "baseline_transformer"\nd_model: 512\nn_layers: 6\nn_heads: 8\nd_ff: 2048\n'
        'dropout: 0.1\nvocab_size: 50257\nmax_seq_len: 1024\n')

    # FIX BUG 4: was flat key hybrid_kernel_size; train.py reads cfg.model.hybrid.type
    # and cfg.model.hybrid.conv_kernel_size from a NESTED hybrid: block
    write_config(f'{BASE}/model/hybrid.yaml',
        'name: "hybrid_transformer"\nd_model: 512\nn_layers: 6\nn_heads: 8\nd_ff: 2048\n'
        'dropout: 0.1\nvocab_size: 50257\nmax_seq_len: 1024\n\n'
        'hybrid:\n'
        '  type: "conv_before_attn"   # Options: conv_before_attn | gated_conv_ffn | interleaved\n'
        '  conv_kernel_size: 3\n')

    # ── Ensure hybrid model __init__.py exists ─────────────────────────────
    hybrid_init = f'{REPO}/core_ml/models/hybrid/__init__.py'
    os.makedirs(os.path.dirname(hybrid_init), exist_ok=True)
    if not os.path.exists(hybrid_init):
        open(hybrid_init, 'w').close()
        print(f'  created core_ml/models/hybrid/__init__.py')

    print('\n✅ All configs written and verified')

except Exception as e:
    print(f'\n❌ Config setup FAILED: {e}')
    traceback.print_exc()

In [ ]:
# ── 0D  Verify config names match train.py dispatch table ────────────────────
# This cell catches any remaining name mismatches BEFORE a long training run fails.

import yaml, os, traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'
BASE = f'{REPO}/core_ml/configs'

# Ground-truth: what train.py accepts in each dispatch block
VALID_ATTN = {'vanilla_mha', 'sliding_window', 'gqa', 'relu_attention', 'sparse_attention'}
VALID_POS  = {'sinusoidal', 'rope', 'alibi', 'relative'}

all_ok = True

try:
    print('Checking attention configs...')
    for fname in os.listdir(f'{BASE}/attention'):
        if not fname.endswith('.yaml'): continue
        path = f'{BASE}/attention/{fname}'
        with open(path) as f:
            cfg = yaml.safe_load(f)
        name = cfg.get('name', '???')
        if name in VALID_ATTN:
            print(f'  ✅ {fname:30s}  name={name}')
        else:
            print(f'  ❌ {fname:30s}  name={name!r}  NOT in {VALID_ATTN}')
            all_ok = False

    print('\nChecking positional configs...')
    for fname in os.listdir(f'{BASE}/positional'):
        if not fname.endswith('.yaml'): continue
        path = f'{BASE}/positional/{fname}'
        with open(path) as f:
            cfg = yaml.safe_load(f)
        name = cfg.get('name', '???')
        if name in VALID_POS:
            print(f'  ✅ {fname:30s}  name={name}')
        else:
            print(f'  ❌ {fname:30s}  name={name!r}  NOT in {VALID_POS}')
            all_ok = False

    print('\nChecking hybrid model config...')
    with open(f'{BASE}/model/hybrid.yaml') as f:
        hcfg = yaml.safe_load(f)
    hybrid_block = hcfg.get('hybrid', None)
    if isinstance(hybrid_block, dict) and 'type' in hybrid_block and 'conv_kernel_size' in hybrid_block:
        print(f'  ✅ hybrid.yaml  nested hybrid block OK: type={hybrid_block["type"]}, conv_kernel_size={hybrid_block["conv_kernel_size"]}')
    else:
        print(f'  ❌ hybrid.yaml  missing nested hybrid: block — got: {hybrid_block}')
        all_ok = False

    print()
    if all_ok:
        print('✅ All config names are valid')
    else:
        print('❌ Some configs have wrong names — re-run cell 0C to fix')

except Exception as e:
    print(f'❌ Verification FAILED: {e}')
    traceback.print_exc()

In [ ]:
# ── 0E  W&B login ─────────────────────────────────────────────────────────────
try:
    import wandb
    wandb.login()  # paste API key from https://wandb.ai/authorize
    print('✅ W&B logged in')
except Exception as e:
    print(f'❌ W&B login failed: {e}')
    import traceback; traceback.print_exc()

In [ ]:
# ── 0F  Validate imports (catches missing files before training starts) ───────
import traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'
import sys; sys.path.insert(0, REPO)
import os; os.chdir(REPO)

checks = [
    ('core_ml.data.dataset',                    'LanguageModelingDataset'),
    ('core_ml.models.transformer',              'Transformer'),
    ('core_ml.models.blocks',                   'TransformerBlock'),
    ('core_ml.models.ffn',                      'FeedForward'),
    ('core_ml.models.attention.vanilla_attention', 'MultiHeadAttention'),
    ('core_ml.models.attention.sliding_window', 'SlidingWindowAttention'),
    ('core_ml.models.attention.gqa',            'GroupedQueryAttention'),
    ('core_ml.models.attention.relu_attention', 'ReLUAttention'),
    ('core_ml.models.attention.Sparse_attention', 'SparseAttention'),
    ('core_ml.models.positional.Sinusoidal',    'SinusoidalPositionalEncoding'),
    ('core_ml.models.positional.Rope',          'RotaryPositionalEmbedding'),
    ('core_ml.models.positional.Alibi',         'ALiBiPositionalBias'),
    ('core_ml.models.positional.RelativeBias',  'RelativePositionalBias'),
    ('core_ml.models.hybrid.hybrid_blocks',     'ConvBeforeAttnBlock'),
    ('core_ml.train.train',                     'build_model'),
    ('core_ml.train.trainer',                   'Trainer'),
    ('core_ml.train.dataset',                   'prepare_dataloaders'),
]

all_ok = True
for module_path, symbol in checks:
    try:
        mod = __import__(module_path, fromlist=[symbol])
        getattr(mod, symbol)
        print(f'  ✅ {module_path}.{symbol}')
    except Exception as e:
        print(f'  ❌ {module_path}.{symbol}  →  {e}')
        all_ok = False

print()
if all_ok:
    print('✅ All imports OK — safe to start training')
else:
    print('❌ Fix the imports above before running training cells')

## 1  Core ML Training Experiments

Each cell runs one training configuration.  
Output streams live to the cell.  Any crash prints the full traceback immediately.

In [ ]:
# ── Helper: run a training command and surface errors clearly ─────────────────
import subprocess, sys, os, traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'

def run_experiment(name, overrides):
    """
    Runs `python core_ml/train/train.py <overrides>` from the repo root.
    Streams stdout/stderr live and raises on non-zero exit.
    """
    cmd = [sys.executable, 'core_ml/train/train.py'] + overrides
    print(f'\n{"="*60}')
    print(f'  Starting: {name}')
    print(f'  Command:  {" ".join(cmd)}')
    print(f'{"="*60}\n')

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,   # merge stderr into stdout
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()

        if proc.returncode != 0:
            print(f'\n❌ FAILED: {name}  (exit code {proc.returncode})')
            print('    → Scroll up to find the Python traceback in the output above.')
            return False
        else:
            print(f'\n✅ DONE: {name}')
            return True

    except Exception as e:
        print(f'❌ Could not launch process: {e}')
        traceback.print_exc()
        return False

print('✅ run_experiment helper defined')

In [ ]:
# ── 1.1  Baseline: Vanilla MHA + Sinusoidal ───────────────────────────────────
run_experiment('vanilla_sinusoidal', [
    'attention=vanilla',
    'positional=sinusoidal',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=baseline_vanilla_sin',
])

In [ ]:
# ── 1.2  Sliding Window + Sinusoidal ─────────────────────────────────────────
run_experiment('sliding_window_sinusoidal', [
    'attention=sliding_window',
    'positional=sinusoidal',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=attn_sliding_sin',
])

In [ ]:
# ── 1.3  GQA + Sinusoidal ────────────────────────────────────────────────────
run_experiment('gqa_sinusoidal', [
    'attention=gqa',
    'positional=sinusoidal',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=attn_gqa_sin',
])

In [ ]:
# ── 1.4  ReLU Attention + Sinusoidal ─────────────────────────────────────────
run_experiment('relu_attention_sinusoidal', [
    'attention=relu',
    'positional=sinusoidal',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=attn_relu_sin',
])

In [ ]:
# ── 1.5  Sparse Attention + Sinusoidal ────────────────────────────────────────
run_experiment('sparse_attention_sinusoidal', [
    'attention=sparse',
    'positional=sinusoidal',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=attn_sparse_sin',
])

In [ ]:
# ── 1.6  Vanilla + RoPE ──────────────────────────────────────────────────────
run_experiment('vanilla_rope', [
    'attention=vanilla',
    'positional=rope',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=pos_rope',
])

In [ ]:
# ── 1.7  Vanilla + ALiBi ─────────────────────────────────────────────────────
run_experiment('vanilla_alibi', [
    'attention=vanilla',
    'positional=alibi',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=pos_alibi',
])

In [ ]:
# ── 1.8  Vanilla + Relative Bias ──────────────────────────────────────────────
run_experiment('vanilla_relative', [
    'attention=vanilla',
    'positional=relative',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=pos_relative',
])

In [ ]:
# ── 1.9  Hybrid: Conv-before-Attn ─────────────────────────────────────────────
# Set these to whichever attention + positional gave the best val perplexity above.
BEST_ATTN = 'sliding_window'   # ← update to your best
BEST_POS  = 'alibi'            # ← update to your best

run_experiment('hybrid_conv_before_attn', [
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=conv_before_attn',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_conv_before',
])

In [ ]:
# ── 1.10  Hybrid: Gated Conv FFN ──────────────────────────────────────────────
run_experiment('hybrid_gated_conv_ffn', [
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=gated_conv_ffn',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_gated_conv',
])

In [ ]:
# ── 1.11  Hybrid: Interleaved ─────────────────────────────────────────────────
run_experiment('hybrid_interleaved', [
    f'attention={BEST_ATTN}',
    f'positional={BEST_POS}',
    'model=hybrid',
    'model.hybrid.type=interleaved',
    'dataset.batch_size=8',
    'dataset.num_workers=2',
    'experiment_name=hybrid_interleaved',
])

## 2  Benchmark All Checkpoints

Run after all (or some) training cells complete.  
Reads checkpoints from `outputs/` and logs perplexity + latency to W&B.

**BUG FIXED**: original notebook patched `core_ml/experiments/benchmark.py` which  
does not exist — the file is at `core_ml/benchmark.py`.

In [ ]:
# ── Helper: run benchmark for one config ──────────────────────────────────────
import subprocess, sys, os, traceback

REPO = '/content/SAiDL-Summer-Assignment-2026'

def run_benchmark(attn, pos, extra_overrides=None):
    """
    Runs `python core_ml/benchmark.py` for the given attention+positional combo.
    FIX: correct script path is core_ml/benchmark.py, NOT core_ml/experiments/benchmark.py
    """
    overrides = [
        f'attention={attn}',
        f'positional={pos}',
        'dataset.batch_size=4',
        'dataset.num_workers=2',
    ] + (extra_overrides or [])

    # FIX BUG 7: correct path is core_ml/benchmark.py
    cmd = [sys.executable, 'core_ml/benchmark.py'] + overrides
    name = f'bench_{attn}_{pos}'

    print(f'\n{"="*60}')
    print(f'  Benchmarking: {name}')
    print(f'  Command: {" ".join(cmd)}')
    print(f'{"="*60}\n')

    try:
        proc = subprocess.Popen(
            cmd, cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()

        if proc.returncode != 0:
            print(f'\n❌ FAILED: {name}  (exit code {proc.returncode})')
            print('    → Scroll up to see the Python traceback.')
            return False
        else:
            print(f'\n✅ DONE: {name}')
            return True

    except Exception as e:
        print(f'❌ Could not launch benchmark: {e}')
        traceback.print_exc()
        return False

print('✅ run_benchmark helper defined')

In [ ]:
# ── 2.1  Benchmark all standard attention × sinusoidal runs ───────────────────
configs = [
    ('vanilla',        'sinusoidal'),
    ('sliding_window', 'sinusoidal'),
    ('gqa',            'sinusoidal'),
    ('relu',           'sinusoidal'),
    ('sparse',         'sinusoidal'),
]

for attn, pos in configs:
    run_benchmark(attn, pos)

In [ ]:
# ── 2.2  Benchmark positional encoding variants ────────────────────────────────
pos_configs = [
    ('vanilla', 'rope'),
    ('vanilla', 'alibi'),
    ('vanilla', 'relative'),
]

for attn, pos in pos_configs:
    run_benchmark(attn, pos)

In [ ]:
# ── 2.3  Benchmark hybrid models ──────────────────────────────────────────────
BEST_ATTN = 'sliding_window'   # ← match what you used in 1.9–1.11
BEST_POS  = 'alibi'

for hybrid_type in ['conv_before_attn', 'gated_conv_ffn', 'interleaved']:
    run_benchmark(BEST_ATTN, BEST_POS, extra_overrides=[
        'model=hybrid',
        f'model.hybrid.type={hybrid_type}',
    ])

In [ ]:
# ── 2.4  Print benchmark results summary ──────────────────────────────────────
import json, os, glob

REPO = '/content/SAiDL-Summer-Assignment-2026'
result_files = sorted(glob.glob(f'{REPO}/benchmark_results/*.json'))

if not result_files:
    print('No benchmark results found yet. Run the benchmark cells above first.')
else:
    print(f'Found {len(result_files)} result files:\n')
    rows = []
    for path in result_files:
        with open(path) as f:
            d = json.load(f)
        rows.append(d)

    # Print as aligned table
    cols = ['attention', 'positional', 'n_params_M', 'ppl_seq512', 'ppl_seq1024', 'latency_ms', 'tokens_per_sec']
    header = '  '.join(f'{c:20s}' for c in cols)
    print(header)
    print('-' * len(header))
    for row in rows:
        line = '  '.join(f'{str(row.get(c, "—")):20s}' for c in cols)
        print(line)

## 3  Download outputs

In [ ]:
import os
REPO = '/content/SAiDL-Summer-Assignment-2026'

os.system(f'cd {REPO} && zip -r /content/core_ml_outputs.zip outputs benchmark_results 2>/dev/null || true')

from google.colab import files
files.download('/content/core_ml_outputs.zip')
print('Download started.')